# Trump Tweet Fine-Tuning on DistilGPT-2
Fine-tune DistilGPT-2 on Trump's Legacy dataset to generate Trump-style tweets.

In [ ]:
# Install required packages
!pip install -q torch transformers datasets pandas kaggle

In [ ]:
import torch
import pandas as pd
from pathlib import Path

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Set up Kaggle API from Colab Secrets
from google.colab import userdata
kaggle_username = userdata.get('KAGGLE_USERNAME')
kaggle_key = userdata.get('KAGGLE_KEY')

# Save credentials to ~/.kaggle/kaggle.json
import os
import json
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(kaggle_dir / 'kaggle.json', 0o600)
print('Kaggle API configured')

In [ ]:
# Download Trump's Legacy dataset from Kaggle
import subprocess
dataset_path = Path('/tmp/trump_data')
dataset_path.mkdir(exist_ok=True)

print('Downloading Trump\'s Legacy dataset...')
subprocess.run(['kaggle', 'datasets', 'download', '-d', 'zusmani/trumps-legacy', '-p', str(dataset_path)], check=True)

# Extract zip
import zipfile
zip_file = list(dataset_path.glob('*.zip'))[0]
with zipfile.ZipFile(zip_file, 'r') as z:
    z.extractall(dataset_path)
print(f'Dataset extracted to {dataset_path}')
print('Files:', list(dataset_path.glob('*.csv'))[:3])

In [ ]:
# Load tweets dataset
csv_files = list(dataset_path.glob('*.csv'))
df = pd.read_csv(csv_files[0])
print(f'Loaded {len(df)} tweets')
print(f'Columns: {df.columns.tolist()}')
print(f'\nFirst 3 tweets:')
print(df['text'].head(3).values)

In [ ]:
# Data exploration
print(f'Total tweets: {len(df)}')
print(f'Avg tweet length: {df["text"].str.len().mean():.0f} chars')
print(f'Min/Max length: {df["text"].str.len().min()}/{df["text"].str.len().max()}')
print(f'\nSample Trump tweets:')
for i, tweet in enumerate(df['text'].sample(3).values, 1):
    print(f'{i}. {tweet[:80]}...' if len(tweet) > 80 else f'{i}. {tweet}')

In [ ]:
# Clean tweets: remove URLs, @mentions, and extra whitespace
import re

def clean_tweet(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_tweet)
# Remove empty tweets
df = df[df['text'].str.len() > 10]
print(f'After cleaning: {len(df)} tweets')
print(f'Sample cleaned tweet: {df["text"].iloc[0][:80]}...')

In [ ]:
# Prepare tokenizer and encode tweets
from transformers import AutoTokenizer

model_name = 'distilgpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize all tweets
encodings = tokenizer(df['text'].tolist(), truncation=True, max_length=128, padding='max_length', return_tensors='pt')
print(f'Tokenized {len(encodings["input_ids"])} tweets')
print(f'Token shape: {encodings["input_ids"].shape}')

In [ ]:
# Create train/validation split (80/20)
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Load pre-trained DistilGPT-2
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
print(f'Model loaded: {model_name}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')
model.train()

In [ ]:
# Training configuration
from torch.optim import AdamW

num_epochs = 3
learning_rate = 5e-5
optimizer = AdamW(model.parameters(), lr=learning_rate)

print(f'Epochs: {num_epochs}, LR: {learning_rate}, Batch size: {batch_size}')

In [ ]:
# Fine-tuning loop
import time

best_val_loss = float('inf')
checkpoint_dir = Path('/tmp/trump_model_checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

for epoch in range(num_epochs):
    # Training
    train_loss = 0
    model.train()
    for batch_idx, (input_ids, attention_mask) in enumerate(train_loader):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}')
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for input_ids, attention_mask in val_loader:
            input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=input_ids)
            val_loss += outputs.loss.item()
    
    avg_val_loss = val_loss / len(val_loader)
    print(f'Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}')
    
    # Save best checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(checkpoint_dir / f'best_model')
        print(f'  -> Saved best model (val loss: {avg_val_loss:.4f})')

In [ ]:
# Load best checkpoint and generate samples
from transformers import TextGenerationPipeline

model.load_state_dict(torch.load(checkpoint_dir / 'best_model' / 'pytorch_model.bin', map_location=device))
model.eval()

# Generate text with different seed prompts
prompts = ['Make America', 'I will', 'The fake news', 'Beautiful', 'Total disaster']
print('Generated Trump-style tweets:\n')

with torch.no_grad():
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
        output = model.generate(input_ids, max_length=80, num_beams=1, do_sample=True, top_k=50, top_p=0.95, temperature=0.7)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        print(f'Prompt: "{prompt}"')
        print(f'Generated: "{generated_text}"\n')

In [ ]:
# Optional: Compare perplexity before/after fine-tuning
import math

print('Perplexity Analysis (lower is better)\n')

# Fine-tuned model perplexity
finetuned_loss = best_val_loss
finetuned_ppl = math.exp(finetuned_loss)
print(f'Fine-tuned model validation perplexity: {finetuned_ppl:.2f}')

# Load original DistilGPT-2 for comparison
original_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
original_model.eval()
orig_val_loss = 0
with torch.no_grad():
    for input_ids, attention_mask in val_loader:
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        outputs = original_model(input_ids, attention_mask=attention_mask, labels=input_ids)
        orig_val_loss += outputs.loss.item()
orig_val_loss /= len(val_loader)
orig_ppl = math.exp(orig_val_loss)

print(f'Original DistilGPT-2 validation perplexity: {orig_ppl:.2f}')
print(f'\nImprovement: {orig_ppl - finetuned_ppl:.2f} perplexity points ({(orig_ppl/finetuned_ppl - 1)*100:.1f}% reduction)')